# Phase 5 (Legal Statute Identification) - Fine-tuning the Phase 1 models

One notebook, several experiments run back to back (`RUNS` below), one run per GPU. Every run:

1. loads a Phase 1 checkpoint (Model 2 / 3 / 4 / 5) and puts a **multi-label document head** on it: the document is
   tokenized, a part of it is cut into chunks of `chunk_len` tokens (`section`, `n_chunks`), the backbone reads every
   chunk, each chunk's tokens are mean-pooled into one vector, the chunk vectors are pooled into one document vector
   (`pool`: mean, or learned attention weights), and a linear layer gives one score per statute (100 statutes,
   independent sigmoid outputs, binary cross-entropy),
2. adapts it with the run's method (`full`, `partial`, `lora`, `rslora`, `lorma`, `bitfit`, `kd`),
3. trains on train.json and keeps the best epoch by **Macro-F1 on dev** (same scoring as `evaluate_phase5.py`;
   statutes predicted when their probability is above one global threshold, chosen on dev),
4. scores the trained model on dev with other parts of the document (`eval_sections`), and predicts test.json ->
   `<run>/submission.zip` (prediction.json at the root).

**Data note.** Documents with the same text are kept once (their statute lists merged). Training documents that are
also in dev or test, and dev documents that are also in test, are dropped.

**Inputs to add on Kaggle:** the HF secret `HF` and one dataset per Phase 1 model (as in Phase 4).
**Outputs:** `/kaggle/working/phase5_log.txt` and `/kaggle/working/phase5_results_bundle.zip` (logs, metrics,
submissions, dev/test logits) - download this one.

The hyperparameters below are starting values, not tuned ones.

In [ ]:
# ============================ CONFIG ============================
MODEL_DIRS = {          # folder (or any file inside it) of each Phase 1 checkpoint; None = auto-discover
    "m2": None,         # Model 2: A-3 decoder, RoPE + GQA          (a3_config.json + a3_model.pt)
    "m3": None,         # Model 3: A-2 GRU encoder-decoder           (gru_config.json + gru_model.pt, or Trainer *.safetensors)
    "m4": "/kaggle/input/datasets/sanskaars22/model-5",   # Model 4: modern decoder (this upload is legal-modern-dec-101m)
    "m5": "/kaggle/input/datasets/sanskaars22/model-4-prefix-lm-labelled-as-m5-in-notebook",   # Model 5: prefix-LM (legal-prefix-lm-101m)
}
DISCOVER_KEYWORDS = {"m2": ["model-2", "model_2", "model2", "dec-rope"], "m3": ["model-3", "model_3", "model3", "gru"],
                     "m4": ["model-4", "model_4", "model4"], "m5": ["model-5", "model_5", "model5", "prefix"]}

COMMON = dict(
    section="head",                # part of the document the model reads: "head" (first tokens), "tail" (last), "middle",
                                   # "head_tail" (first half + last half of the budget), "spread" (n_chunks evenly spaced chunks)
    chunk_len=512,                 # tokens per chunk (<= the model's max_seq_len)
    n_chunks=4,                    # chunks per document: the model reads n_chunks x chunk_len tokens of `section`
                                   # (4 x 512: 78% of the training documents are read completely)
    pool="attn",                   # chunk vectors -> document vector: "mean" or "attn" (learned weights, start as the mean)
    max_train_docs=15000,          # fixed random subset of the ~42k unique training documents (the same for every run); None = all
    dev_select_docs=2500,          # epoch selection on this fixed random part of dev (full dev scored at the end); None = all
    batch_tokens=4096,             # tokens per batch (~8 chunks of 512)
    epochs=3, grad_accum=2,
    lr_backbone=3e-5,              # pretrained weights (full / partial / kd)
    lr_adapter=3e-4,               # LoRA / rsLoRA / LoRMA / BitFit parameters
    lr_head=1e-3,                  # the new classification head (+ attention pooling)
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    rank=8, alpha=16, adapter_targets="attn+mlp",   # LoRA / rsLoRA / LoRMA
    partial_trainable_frac=0.5,    # partial: train the top half of the layers (+ head)
    head_dropout=0.1,
    class_weighting="none",        # BCE positive weight per statute: "none", "pos_weight" (n_neg / n_pos) or
                                   # "sqrt_pos_weight" (its square root)
    attn="bidir",                  # decoders: "bidir" (no causal mask during fine-tuning) or "causal"
    eval_sections=["head:1", "middle:1", "tail:1", "head:8"],   # after training: dev scores reading other parts
                                   # ("name:K" = K chunks; "head:8" = the first 8 x chunk_len tokens ~ the whole document)
    test_sections=["head:8"],      # also predict test with these (saved in logits.pt for post-processing; [] = off)
    kd_temperature=2.0, kd_alpha=0.5,
    student_layers=None,           # keep only the lowest N layers of the checkpoint (any method; None = all layers)
    save_teacher_logits=False,     # True for a KD teacher (automatic when a run of RUNS names this run as its teacher)
    patience=None,                 # stop after this many epochs without dev improvement (None = off)
    save_weights=True,             # save the trainable parameters of the best epoch
    seed=42,
)

# Techniques (spec 11.2.1 list): full fine-tuning, partial fine-tuning, LoRMA, knowledge distillation.
# m3 (GRU) has no Linear layers: its method is "full", "partial" or "bitfit".
# A KD run is dict(name=..., model=..., method="kd", teacher=<an earlier run's name>, student_layers=N), after its teacher;
# a teacher trained in an earlier session is found as <teacher>/teacher_logits.pt in an input (that session's bundle).
RUNS = [
    dict(name="m5_full",    model="m5", method="full"),
    dict(name="m5_lorma",   model="m5", method="lorma"),
    dict(name="m4_full",    model="m4", method="full"),
    dict(name="m4_lorma",   model="m4", method="lorma"),
    dict(name="m3",         model="m3", method="full"),
    # KD: 4-layer m5 student, teacher m5_full (in this session, or its teacher_logits.pt from an earlier bundle)
    dict(name="m5_kd_student", model="m5", method="kd", teacher="m5_full", student_layers=4),
    dict(name="m2_lorma",   model="m2", method="lorma"),
    dict(name="m5_partial", model="m5", method="partial"),
    # long-document representation (compare with m5_lorma): first 512 tokens only, on all training documents
    # (about the same training tokens as 15k documents x 4 chunks)
    dict(name="m5_lorma_head1_alldocs", model="m5", method="lorma", n_chunks=1, max_train_docs=None),
    # class weighting (compare with m5_lorma)
    dict(name="m5_lorma_sqrtpw", model="m5", method="lorma", class_weighting="sqrt_pos_weight"),
]
RUN_ONLY = ["m4_full", "m4_lorma"]   # runs behind the best submission; None = every run in RUNS
# session B: no extra reading views (session A already compares them; they cost ~1 h per 100M run)
COMMON.update(eval_sections=[], test_sections=[])
SMOKE = False              # True: 200 training documents, 64 dev documents, 1 epoch per selected run (checks loading,
                           # training, predicting and zipping); outputs go to /kaggle/working/smoke/

In [ ]:
import os, re, json, glob, math, time, random, zipfile, hashlib, gc, platform, threading
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("p5_work")
if SMOKE: WORK = os.path.join(WORK, "smoke")
os.makedirs(WORK, exist_ok=True)
LOG_PATH = os.path.join(WORK, "phase5_log.txt")
_log = open(LOG_PATH, "a", encoding="utf-8")

_log_lock = threading.Lock()
def log(*a):
    s = " ".join(str(x) for x in a)
    t = threading.current_thread().name
    if t.startswith("gpu"): s = "\n".join(f"[{t}] {l}" for l in s.split("\n"))   # parallel runs: which GPU worker
    with _log_lock:
        print(s, flush=True); _log.write(s + "\n"); _log.flush()

def section(t):
    log("\n" + "=" * 100); log(t); log("=" * 100)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
section(f"Phase 5 fine-tuning - {time.strftime('%Y-%m-%d %H:%M:%S')}")
log(f"torch {torch.__version__} | python {platform.python_version()} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.device_count()} GPU)" if DEVICE.type == "cuda" else "")
    + f" | amp {AMP_DTYPE}")
log("COMMON =", json.dumps(COMMON))

In [ ]:
# ---- data: Phase_5_LSI from the official dataset ----
def load_phase5_files():
    local = os.environ.get("P5_LOCAL_DATA")                       # local testing only
    if local:
        return {f: os.path.join(r, f) for r, _, fs in os.walk(local) for f in fs}
    from huggingface_hub import HfApi, hf_hub_download
    token, why = os.environ.get("HF_TOKEN"), "HF_TOKEN env var not set"
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF")
        except Exception as e:
            why = f"Kaggle secret 'HF' not readable ({e!r})"
    assert token, (f"no Hugging Face token: {why}. In the Kaggle editor: Add-ons -> Secrets -> add/attach a secret "
                   "named exactly HF (value = your hf_... token) and make sure its checkbox is ON for this notebook.")
    try:
        log(f"HF login ok: {HfApi().whoami(token=token)['name']}")
    except Exception as e:
        raise RuntimeError(f"the HF token was found but is rejected by Hugging Face ({e!r}); "
                           "create a new read token at huggingface.co/settings/tokens") from None
    repo = "Exploration-Lab/CS-781-Capstone"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset", token=token) if f.startswith("Phase_5_LSI/")]
    return {os.path.basename(f): hf_hub_download(repo, f, repo_type="dataset", token=token) for f in files}

P5 = load_phase5_files()
read = lambda n: json.load(open(P5[n], encoding="utf-8"))

DOCS = {}           # document id -> dict(text, labels (sorted statute ids; None for test), split)
TEST_IDS = []
for x in read("test.json"):
    assert x["id"] not in DOCS, f"duplicate test id {x['id']}"
    DOCS[x["id"]] = dict(text=x["text"], labels=None, split="test"); TEST_IDS.append(x["id"])
TEST_IDS_SET = set(TEST_IDS)
_test_texts = {DOCS[d]["text"] for d in TEST_IDS}

def _add(rows, split, drop_texts):
    """Keeps each text once (statute lists of duplicates merged); drops texts in drop_texts. Returns (kept ids, stats)."""
    seen, ids, st = {}, [], Counter()
    for x in rows:
        if x["text"] in drop_texts: st["dropped"] += 1; continue
        if x["text"] in seen:
            st["duplicate"] += 1
            d = DOCS[seen[x["text"]]]
            if set(x["labels"]) != set(d["labels"]): st["merged"] += 1
            d["labels"] = sorted(set(d["labels"]) | set(x["labels"]))
            continue
        did, k = x["id"], 1
        while did in DOCS: did, k = f"{x['id']}#{k}", k + 1     # the same id with another text: keep both
        seen[x["text"]] = did
        DOCS[did] = dict(text=x["text"], labels=sorted(set(int(l) for l in x["labels"])), split=split); ids.append(did)
    return ids, st

VAL_IDS, _vst = _add(read("dev.json"), "dev", _test_texts)
TRAIN_ALL, _tst = _add(read("train.json"), "train", _test_texts | {DOCS[d]["text"] for d in VAL_IDS})
N_LABELS = 1 + max(l for d in TRAIN_ALL + VAL_IDS for l in DOCS[d]["labels"])

section("DATA")
log(f"test:  {len(TEST_IDS)} documents")
log(f"dev:   kept {len(VAL_IDS)}; dropped {_vst['dropped']} that are test documents, {_vst['duplicate']} duplicates "
    f"({_vst['merged']} with other statutes, merged)")
log(f"train: kept {len(TRAIN_ALL)}; dropped {_tst['dropped']} that are dev / test documents, {_tst['duplicate']} duplicates "
    f"({_tst['merged']} with other statutes, merged)")
for name, ids in (("train", TRAIN_ALL), ("dev", VAL_IDS)):
    c = Counter(l for d in ids for l in DOCS[d]["labels"]); f = np.array([c[l] for l in range(N_LABELS)])
    log(f"{name:5s} {N_LABELS} statutes: {np.mean([len(DOCS[d]['labels']) for d in ids]):.2f} per document; documents per "
        f"statute min {f.min()}, median {int(np.median(f))}, max {f.max()}")
if SMOKE:
    TRAIN_ALL, VAL_IDS = TRAIN_ALL[:200], VAL_IDS[:64]
    log(f"SMOKE: {len(TRAIN_ALL)} train / {len(VAL_IDS)} dev documents")
    _keep = set(TRAIN_ALL) | set(VAL_IDS) | TEST_IDS_SET
    DOCS = {d: x for d, x in DOCS.items() if d in _keep}

In [ ]:
# ---- Phase 1 byte-level BPE (same algorithm and ids as LegalBPETokenizer), with character offsets ----
WORD_RE = re.compile(r"\S+")
_SHIFT = 20
_EOW = (1 << _SHIFT) - 1

class OffsetBPE:
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        self.itos = d["itos"]
        sym2id = {tuple(s): i for i, s in enumerate(self.itos) if isinstance(s, list)}
        to_id = lambda t: _EOW if tuple(t) == (256,) else sym2id[tuple(t)]
        self.ranks = {(to_id(l) << _SHIFT) | to_id(r): sym2id[tuple(l) + tuple(r)] for l, r in d["merges"]}
        self.nbytes = [sum(b != 256 for b in s) if isinstance(s, list) else 0 for s in self.itos]
        self.eow_id = len(self.itos)               # LegalBPETokenizer: a bare end-of-word symbol gets id len(itos)
        self.vocab_size = len(self.itos) + 1
        self.cache = {}

    def tokenize_word(self, word):
        out = self.cache.get(word)
        if out is not None: return out
        w = [b + 4 for b in word.encode("utf-8")] + [_EOW]
        ranks = self.ranks
        while len(w) > 1:
            best = None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best is None or n < best[0]): best = (n, x, y)
            if best is None: break
            n, a, b = best; i, res = 0, []
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: res.append(n); i += 2
                else: res.append(w[i]); i += 1
            w = res
        self.cache[word] = w
        return w

    def encode(self, text):
        """-> ids, char_start, char_end (np arrays). A bare end-of-word token has an empty span at the word end."""
        ids, cs, ce = [], [], []
        for m in WORD_RE.finditer(text):
            word, base = m.group(), m.start()
            b2c = []
            for ci, ch in enumerate(word): b2c.extend([ci] * len(ch.encode("utf-8")))
            p = 0
            for t in self.tokenize_word(word):
                nb = 0 if t == _EOW else self.nbytes[t]
                ids.append(self.eow_id if t == _EOW else t)
                if nb == 0:
                    cs.append(base + len(word)); ce.append(base + len(word))
                else:
                    cs.append(base + b2c[p]); ce.append(base + b2c[p + nb - 1] + 1); p += nb
        return np.array(ids, np.int64), np.array(cs, np.int64), np.array(ce, np.int64)

# ---- From-scratch attention (no nn.MultiheadAttention / F.scaled_dot_product_attention), as in Phase 1 ----
# out = softmax(q k^T * scale, masked) v, computed in query blocks with the softmax statistics kept per row
# (log-sum-exp), so the full (T, S) score matrix is never stored. The backward pass is written by hand and
# recomputes each block's probabilities from the saved log-sum-exp:
#   P = exp(S - lse),  dV = P^T dO,  dP = dO V^T,  dS = P * (dP - rowsum(dO * O)),  dQ = dS K * scale,  dK = dS^T Q * scale
def _block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset):
    s = torch.matmul(q_blk, k[:, :, :kend].transpose(-2, -1)).float() * scale
    if attn_mask is not None:
        s = s.masked_fill(~attn_mask[..., i:j, :kend], float("-inf"))
    if is_causal:
        rows = torch.arange(i, j, device=s.device)[:, None] + offset
        s = s.masked_fill(torch.arange(kend, device=s.device)[None, :] > rows, float("-inf"))
    return s


class BlockwiseAttention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, attn_mask, is_causal, scale, block):
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        out = torch.empty_like(q)
        lse = torch.empty(q.shape[:-1], dtype=torch.float32, device=q.device)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            s = _block_scores(q[:, :, i:j], k, scale, attn_mask, is_causal, i, j, kend, offset)
            m = s.amax(-1, keepdim=True)
            p = torch.exp(s - m)
            l = p.sum(-1, keepdim=True)
            out[:, :, i:j] = torch.matmul((p / l).to(v.dtype), v[:, :, :kend])
            lse[:, :, i:j] = (m + l.log()).squeeze(-1)
        ctx.save_for_backward(q, k, v, out, lse, attn_mask)
        ctx.is_causal, ctx.scale, ctx.block = is_causal, scale, block
        return out

    @staticmethod
    def backward(ctx, d_out):
        q, k, v, out, lse, attn_mask = ctx.saved_tensors
        is_causal, scale, block = ctx.is_causal, ctx.scale, ctx.block
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        d_out = d_out.contiguous()
        dq = torch.empty_like(q)
        dk = torch.zeros(k.shape, dtype=torch.float32, device=k.device)
        dv = torch.zeros(v.shape, dtype=torch.float32, device=v.device)
        delta = (d_out.float() * out.float()).sum(-1, keepdim=True)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            q_blk, do_blk = q[:, :, i:j], d_out[:, :, i:j]
            p = torch.exp(_block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset) - lse[:, :, i:j, None])
            dv[:, :, :kend] += torch.matmul(p.to(v.dtype).transpose(-2, -1), do_blk).float()
            dp = torch.matmul(do_blk, v[:, :, :kend].transpose(-2, -1)).float()
            ds = (p * (dp - delta[:, :, i:j])).to(q.dtype)
            dq[:, :, i:j] = torch.matmul(ds, k[:, :, :kend]) * scale
            dk[:, :, :kend] += torch.matmul(ds.transpose(-2, -1), q_blk).float() * scale
        return dq, dk.to(k.dtype), dv.to(v.dtype), None, None, None, None


def scratch_attention(q, k, v, attn_mask=None, is_causal=False, scale=None, block=128):
    """q (B, H, T, d), k/v (B, H_kv, S, d) with H_kv dividing H (GQA). attn_mask: bool (True = may attend)."""
    scale = q.size(-1) ** -0.5 if scale is None else scale
    B, H, T, d = q.shape
    n_rep = H // k.size(1)
    if n_rep > 1 and not is_causal:
        q = q.reshape(B, k.size(1), n_rep * T, d)
        if attn_mask is not None:
            attn_mask = attn_mask.repeat(1, 1, n_rep, 1)
        out = BlockwiseAttention.apply(q, k, v, attn_mask, False, scale, block)
        return out.reshape(B, H, T, d)
    if n_rep > 1:
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
    return BlockwiseAttention.apply(q, k, v, attn_mask, is_causal, scale, block)


def attend(q, k, v, key_keep, causal, scale=None):
    """causal or bidirectional attention; key_keep (B, T) bool marks real tokens (None = no padding)."""
    if key_keep is None:
        return scratch_attention(q, k, v, is_causal=causal, scale=scale)
    T = q.size(-2)
    allowed = key_keep[:, None, None, :].expand(-1, 1, T, -1)
    if causal:
        allowed = allowed & torch.ones(T, T, dtype=torch.bool, device=q.device).tril()
    allowed = allowed | torch.eye(T, dtype=torch.bool, device=q.device)   # padding rows see themselves (no NaN)
    return scratch_attention(q, k, v, attn_mask=allowed, scale=scale)


def rope_cache(T, d_head, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    ang = torch.outer(torch.arange(T, dtype=torch.float32), inv_freq)
    ang = torch.cat([ang, ang], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
# ---- Phase 1 backbones as encoders. Parameter names match the Phase 1 checkpoints exactly. ----
class A3Attention(nn.Module):              # Model 2 (A-3 CausalSelfAttention: MHA/GQA, RoPE or sinusoidal)
    def __init__(self, c):
        super().__init__()
        self.n_heads, self.n_kv, self.d_head = c["n_heads"], c["n_kv_heads"], c["d_model"] // c["n_heads"]
        self.scale = c.get("attention_scale") or self.d_head ** -0.5
        self.q_proj = nn.Linear(c["d_model"], self.n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.v_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.out_proj = nn.Linear(self.n_heads * self.d_head, c["d_model"], bias=False)
        self.use_rope = c.get("pos_emb_type") == "rope"
        if self.use_rope:
            cos, sin = rope_cache(c["max_seq_len"], self.d_head, c.get("rope_base", 10000.0))
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, key_keep, causal):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        if self.use_rope:
            cos, sin = self.rope_cos[:T][None, None].to(q.dtype), self.rope_sin[:T][None, None].to(q.dtype)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = attend(q, k, v, key_keep, causal, self.scale)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, -1))


class A3Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm_first = c.get("norm_first", False)
        self.ln1 = nn.LayerNorm(c["d_model"])
        self.attn = A3Attention(c)
        self.ln2 = nn.LayerNorm(c["d_model"])
        self.ffn = nn.Module()
        self.ffn.net = nn.Sequential(nn.Linear(c["d_model"], c["d_ff"]), nn.GELU(), nn.Linear(c["d_ff"], c["d_model"]), nn.Dropout(0.0))

    def forward(self, x, key_keep, causal):
        if self.norm_first:
            x = x + self.attn(self.ln1(x), key_keep, causal)
            return x + self.ffn.net(self.ln2(x))
        x = self.ln1(x + self.attn(x, key_keep, causal))
        return self.ln2(x + self.ffn.net(x))


class A3Backbone(nn.Module):
    kind = "a3"
    def __init__(self, c):
        super().__init__()
        assert c.get("attention_type", "standard") == "standard", "only the standard (MHA/GQA) A-3 attention is supported"
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        if c.get("pos_emb_type") == "sinusoidal":
            pos = torch.arange(c["max_seq_len"], dtype=torch.float32)[:, None]
            div = torch.exp(torch.arange(0, c["d_model"], 2, dtype=torch.float32) * (-math.log(10000.0) / c["d_model"]))
            pe = torch.zeros(c["max_seq_len"], c["d_model"]); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([A3Block(c) for _ in range(c["n_layers"])])
        self.ln_f = nn.LayerNorm(c["d_model"])

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        if hasattr(self, "pos_emb"): x = x + self.pos_emb[:ids.size(1)][None]
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.ln_f(x)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight


class ModernAttention(A3Attention):         # Models 4/5 GQAttention (always RoPE), same projections
    def __init__(self, c):
        super().__init__({**c, "pos_emb_type": "rope", "attention_scale": None})


class ModernBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.attn = ModernAttention(c)
        self.mlp_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.mlp = nn.Module()
        self.mlp.gate = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.up = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.down = nn.Linear(c["d_ff"], c["d_model"], bias=False)

    def forward(self, x, key_keep, causal):
        x = x + self.attn(self.attn_norm(x), key_keep, causal)
        h = self.mlp_norm(x)
        return x + self.mlp.down(F.silu(self.mlp.gate(h)) * self.mlp.up(h))


class ModernBackbone(nn.Module):
    kind = "modern"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        self.blocks = nn.ModuleList([ModernBlock(c) for _ in range(c["n_layers"])])
        self.final_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.final_norm(x)


class GRUBackbone(nn.Module):               # Model 3: the bidirectional GRU encoder of the A-2 seq2seq model
    kind = "gru"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, 2 * c["hidden"]
        self.embedding = nn.Embedding(c["vocab_size"], c["emb_dim"])
        self.enc_rnn = nn.GRU(c["emb_dim"], c["hidden"], num_layers=c["enc_layers"], batch_first=True, bidirectional=True)

    def forward(self, ids, key_keep=None, causal=False):
        self.enc_rnn.flatten_parameters()
        x = self.embedding(ids)
        if key_keep is None:
            return self.enc_rnn(x)[0]
        lengths = key_keep.sum(1).clamp(min=1).cpu()
        out, _ = self.enc_rnn(pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False))
        return pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))[0]

In [ ]:
# ---- finding and loading the checkpoints ----
def _ckpt_kind(folder):
    names = os.listdir(folder)
    if "a3_config.json" in names: return "a3"
    if "modern_config.json" in names: return "modern"
    if "gru_config.json" in names: return "gru"
    st = [n for n in names if n.endswith(".safetensors")]
    if st:
        from safetensors import safe_open
        with safe_open(os.path.join(folder, st[0]), "pt") as f:
            keys = list(f.keys())
        if any("enc_rnn" in k for k in keys): return "gru"
        if any("attn_norm" in k for k in keys): return "modern"
        if any(".ln1." in k for k in keys): return "a3"
    return None

WANT_KIND = {"m2": "a3", "m3": "gru", "m4": "modern", "m5": "modern"}

def find_checkpoint(key):
    p = MODEL_DIRS.get(key)
    if p:
        folder = p if os.path.isdir(p) else os.path.dirname(p)
        if _ckpt_kind(folder) is None:          # a dataset root: look one level deeper
            subs = [os.path.dirname(f) for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                    if f.endswith(("_config.json", ".safetensors"))]
            subs = [s for s in dict.fromkeys(subs) if "checkpoint-" not in s and _ckpt_kind(s)]
            assert subs, f"no checkpoint found under {p}"
            folder = subs[0]
        return folder
    cands = []
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if f.endswith(("_config.json", ".safetensors")):
            folder = os.path.dirname(f)
            if "checkpoint-" in folder: continue
            if _ckpt_kind(folder) == WANT_KIND[key]: cands.append(folder)
    cands = list(dict.fromkeys(cands))
    hit = [c for c in cands if any(w in c.lower() for w in DISCOVER_KEYWORDS[key])]
    shared = sum(k == WANT_KIND[key] for k in WANT_KIND.values()) > 1   # m4/m5: same format, never guess
    assert hit or (len(cands) == 1 and not shared), \
        f"{key}: cannot pick a checkpoint automatically from {cands}; set MODEL_DIRS['{key}']"
    return (hit or cands)[0]

def _strip(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("module.", "lm.", "model."):
            if k.startswith(pre): k = k[len(pre):]
        out[k] = v
    return out

def load_backbone(key, n_layers=None):
    """Phase 1 checkpoint -> (backbone, info). n_layers < full depth keeps only the lowest layers (KD student)."""
    folder = find_checkpoint(key)
    kind = _ckpt_kind(folder)
    names = os.listdir(folder)
    pt = [n for n in names if n.endswith("_model.pt")]
    if pt:
        sd = torch.load(os.path.join(folder, pt[0]), map_location="cpu", weights_only=True)
    else:
        from safetensors.torch import load_file
        sd = load_file(os.path.join(folder, [n for n in names if n.endswith(".safetensors")][0]))
    sd = _strip(sd)
    cfg_name = {"a3": "a3_config.json", "modern": "modern_config.json", "gru": "gru_config.json"}[kind]
    if cfg_name in names:
        c = json.load(open(os.path.join(folder, cfg_name)))
    elif kind == "gru":                          # Trainer checkpoint without config: read the sizes off the weights
        c = dict(vocab_size=sd["embedding.weight"].shape[0], emb_dim=sd["embedding.weight"].shape[1],
                 hidden=sd["enc_rnn.weight_hh_l0"].shape[1],
                 enc_layers=len([k for k in sd if re.fullmatch(r"enc_rnn\.weight_ih_l\d+", k)]))
    else:
        raise FileNotFoundError(f"{folder}: {cfg_name} is needed to know the number of heads")
    full_layers = c["enc_layers"] if kind == "gru" else c["n_layers"]
    if n_layers:
        assert n_layers <= full_layers
        if kind == "gru":
            c = {**c, "enc_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not re.search(r"_l(\d+)", k) or int(re.search(r"_l(\d+)", k).group(1)) < n_layers}
        else:
            c = {**c, "n_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not k.startswith("blocks.") or int(k.split(".")[1]) < n_layers}
    model = {"a3": A3Backbone, "modern": ModernBackbone, "gru": GRUBackbone}[kind](c)
    own = model.state_dict()
    used = {k: v for k, v in sd.items() if k in own}
    missing = [k for k in own if k not in used]
    assert not missing, f"weights missing from the checkpoint: {missing[:5]}"
    for k, v in used.items():
        assert v.shape == own[k].shape, f"{k}: checkpoint {tuple(v.shape)} vs model {tuple(own[k].shape)}"
    model.load_state_dict(used)
    tok = [os.path.join(folder, n) for n in names if n.startswith("bpe_tokenizer") and n.endswith(".json")]
    for where in (os.path.join(os.path.dirname(folder), "**"), "/kaggle/input/**"):     # next to the weights, then anywhere
        tok = tok or sorted(glob.glob(os.path.join(where, "bpe_tokenizer*.json"), recursive=True))
    assert tok, f"no bpe_tokenizer*.json found for {key}: add the Phase 1 tokenizer as an input"
    info = dict(folder=folder, kind=kind, config=c, full_layers=full_layers, n_layers=n_layers or full_layers,
                unused_keys=len(sd) - len(used), tokenizer=tok[0], prefix_lm=c.get("prefix_lm"))
    return model, info

In [ ]:
# ---- adaptation methods ----
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): h = W0 x + (alpha / r) B A x, A (r x in) random, B (out x r) = 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel & Modi 2025): multiplicative update h = (I + (alpha / r) B A) W0 x, with A (r x out) random,
    B (out x r) = 0 so training starts at W0. Computed as z = W0 x, then z + (alpha / r) B (A z): never forms the d x d matrix."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + (z @ self.A.t().to(z.dtype)) @ self.B.t().to(z.dtype) * self.scale
        return z if self.base.bias is None else z + self.base.bias



class RSLoRALinear(LoRALinear):
    """rsLoRA (Kalajdzievski 2023): LoRA with the update scaled by alpha / sqrt(r) instead of alpha / r, so the update
    does not shrink as the rank grows."""
    def __init__(self, base, r, alpha):
        super().__init__(base, r, alpha)
        self.scale = alpha / math.sqrt(r)


def adapter_target_names(backbone, targets):
    names = []
    for n, m in backbone.named_modules():
        if not isinstance(m, nn.Linear): continue
        is_attn = ".attn." in n
        is_mlp = ".ffn." in n or ".mlp." in n
        if (is_attn and "attn" in targets) or (is_mlp and "mlp" in targets): names.append(n)
    return names

def inject(backbone, cls, targets, r, alpha):
    names = adapter_target_names(backbone, targets)
    for n in names:
        parent, child = backbone.get_submodule(n.rsplit(".", 1)[0]), n.rsplit(".", 1)[1]
        setattr(parent, child, cls(getattr(parent, child), r, alpha))
    return names

In [ ]:
# ---- documents -> token ids (Phase 1 tokenizer), parts of a document -> chunks, scoring (= evaluate_phase5.py) ----
_TOKENS, _TOK_LOCK = {}, threading.Lock()
def get_tokens(tok_path):
    """Every document as one int32 array of token ids, cached per tokenizer file content (computed once per session)."""
    with _TOK_LOCK:
        h = hashlib.md5(open(tok_path, "rb").read()).hexdigest()
        if h not in _TOKENS:
            t0 = time.time(); bpe = OffsetBPE(tok_path); cache = {}
            def enc(text):
                out = []
                for w in text.split():                   # = WORD_RE words
                    t = cache.get(w)
                    if t is None:
                        t = cache[w] = [bpe.eow_id if i == _EOW else i for i in bpe.tokenize_word(w)]
                    out.extend(t)
                return np.array(out or [bpe.eow_id], np.int32)
            _TOKENS[h] = {d: enc(x["text"]) for d, x in DOCS.items()}
            bpe.cache.clear()
            n = np.array([len(_TOKENS[h][d]) for d in TRAIN_ALL])
            log(f"tokenizer {tok_path} (md5 {h[:8]}): {len(DOCS)} documents in {time.time() - t0:.0f}s; train tokens/doc "
                f"median {int(np.median(n))}, p25 {int(np.percentile(n, 25))}, p75 {int(np.percentile(n, 75))}, "
                f"p95 {int(np.percentile(n, 95))}, max {n.max()}; {100 * np.mean(n <= 512):.1f}% <= 512, "
                f"{100 * np.mean(n <= 2048):.1f}% <= 2048, {100 * np.mean(n <= 4096):.1f}% <= 4096")
        return _TOKENS[h]

def parse_section(spec, K):
    """"head" -> ("head", K); "head:8" -> ("head", 8)."""
    name, _, k = spec.partition(":")
    assert name in ("tail", "head", "middle", "head_tail", "spread"), f"unknown section {spec}"
    return name, int(k) if k else K

def select_chunks(t, name, K, L):
    """The part `name` of a document (K x L tokens), as a list of <= K chunks of <= L tokens (in document order)."""
    n, B = len(t), K * L
    if name == "spread":                                 # K chunks of L tokens evenly spaced over the document
        starts = sorted(set(np.linspace(0, max(0, n - L), K).round().astype(int).tolist()))
        return [t[s:s + L] for s in starts]
    if n <= B: region = t
    elif name == "tail": region = t[n - B:]
    elif name == "head": region = t[:B]
    elif name == "middle": region = t[(n - B) // 2:(n - B) // 2 + B]
    else: region = np.concatenate([t[:B // 2], t[n - (B - B // 2):]])     # head_tail
    ends = list(range(len(region), 0, -L))[::-1]          # chunks aligned to the end of the part: the first may be short
    return [region[max(0, e - L):e] for e in ends]

def case_chunks(toks, ids, spec, cfg):
    name, K = parse_section(spec, cfg["n_chunks"])
    return {d: select_chunks(toks[d], name, K, cfg["chunk_len"]) for d in ids}

def gold_matrix(ids):
    Y = np.zeros((len(ids), N_LABELS), bool)
    for i, d in enumerate(ids): Y[i, DOCS[d]["labels"]] = True
    return Y

def decide(P, thr):
    """(n, N_LABELS) probabilities -> 0/1 matrix: statutes above `thr`; a document with none gets its top statute
    (every training document has at least one statute)."""
    Z = P > thr
    Z[np.arange(len(P)), P.argmax(1)] |= ~Z.any(1)
    return Z

def score(Y, Z):
    """Macro-F1 over the statutes (= evaluate_phase5.py, zero_division=0), micro-F1, subset accuracy."""
    tp = (Y & Z).sum(0); fp = (~Y & Z).sum(0); fn = (Y & ~Z).sum(0)
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(1, 2 * tp + fp + fn), 0.0)
    T, FP, FN = tp.sum(), fp.sum(), fn.sum()
    return dict(macro_f1=float(f1.mean()), micro_f1=float(2 * T / max(1, 2 * T + FP + FN)),
                subset_acc=float((Y == Z).all(1).mean()), pred_per_doc=float(Z.sum(1).mean()),
                gold_per_doc=float(Y.sum(1).mean()))

THRESHOLDS = np.round(np.arange(0.02, 0.92, 0.02), 2)
def best_threshold(P, Y):
    """The one global threshold (grid) with the highest dev macro-F1 -> (thr, metrics)."""
    res = [(t, score(Y, decide(P, t))) for t in THRESHOLDS]
    t, m = max(res, key=lambda r: r[1]["macro_f1"])
    return float(t), dict(m, threshold=float(t))

def fmt(m):
    return (f"macro-F1 {m['macro_f1']:.4f} (micro-F1 {m['micro_f1']:.4f}, subset acc {m['subset_acc']:.4f}, "
            f"thr {m['threshold']:.2f}, {m['pred_per_doc']:.2f} predicted / {m['gold_per_doc']:.2f} gold statutes per doc)")

In [ ]:
# ---- batches of documents ----
def case_batches(ids, chunks, budget):
    """Consecutive groups of documents with (number of chunks) x (longest chunk) <= budget tokens (>= 1 per batch)."""
    out, cur, nc, mx = [], [], 0, 0
    for d in ids:
        c, L = len(chunks[d]), max(len(x) for x in chunks[d])
        if cur and (nc + c) * max(mx, L) > budget:
            out.append(cur); cur, nc, mx = [], 0, 0
        cur.append(d); nc += c; mx = max(mx, L)
    if cur: out.append(cur)
    return out

def make_batch(ids, chunks):
    """-> token ids (C, T) of all chunks, key_keep (None if no padding), chunk_case (C,) = index of the document in `ids`."""
    rows = [(b, c) for b, d in enumerate(ids) for c in chunks[d]]
    T = max(len(c) for _, c in rows)
    x = torch.zeros(len(rows), T, dtype=torch.long); keep = torch.zeros(len(rows), T, dtype=torch.bool)
    for r, (_, c) in enumerate(rows):
        x[r, :len(c)] = torch.from_numpy(c.astype(np.int64)); keep[r, :len(c)] = True
    return x, (None if bool(keep.all()) else keep), torch.tensor([b for b, _ in rows])

In [ ]:
# ---- document classifier: chunk mean-pooling -> chunk pooling (mean / attention) -> linear (one score per statute) ----
class CaseClassifier(nn.Module):
    def __init__(self, backbone, dropout, causal, pool, n_out):
        super().__init__()
        self.backbone, self.causal = backbone, causal
        d = backbone.hidden_size
        self.attn_pool = None
        if pool == "attn":                               # score_c = w2 . tanh(W1 v_c); weights = softmax over the doc's chunks
            self.attn_pool = nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.Linear(256, 1))
            nn.init.zeros_(self.attn_pool[2].weight); nn.init.zeros_(self.attn_pool[2].bias)   # starts as the mean
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d, n_out)

    def forward(self, ids, key_keep, chunk_case, n_cases):
        """-> (n_cases, n_out) float32."""
        h = self.backbone(ids, key_keep, self.causal).float()                # (C, T, d)
        if key_keep is None: v = h.mean(1)
        else:
            m = key_keep.float()[..., None]
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)                       # (C, d): one vector per chunk
        if self.attn_pool is None: s = torch.zeros(len(v), device=v.device)
        else: s = self.attn_pool(v).squeeze(-1)
        e = torch.exp(s - s.max())
        w = e / torch.zeros(n_cases, device=v.device).index_add_(0, chunk_case, e)[chunk_case]
        case = torch.zeros(n_cases, v.size(1), device=v.device).index_add_(0, chunk_case, v * w[:, None])
        return self.head(self.drop(case)).float()


def setup_method(model, cfg):
    """Freezes / wraps parameters for the run's method. Returns a list of (param group name, params, lr)."""
    bb, method = model.backbone, cfg["method"]
    for p in bb.parameters(): p.requires_grad = method in ("full", "kd", "partial")
    adapters = []
    if method == "partial":                          # train the top layers only
        (bb.embedding if bb.kind == "gru" else bb.tok_emb).requires_grad_(False)
        if bb.kind == "gru":
            L = bb.c["enc_layers"]; keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for n, p in bb.enc_rnn.named_parameters():
                if int(re.search(r"_l(\d+)", n).group(1)) < keep_from: p.requires_grad = False
        else:
            L = len(bb.blocks); keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for b in bb.blocks[:keep_from]: b.requires_grad_(False)
        log(f"partial: layers {keep_from}..{L - 1} trainable")
    elif method in ("lora", "rslora", "lorma"):
        assert bb.kind != "gru", f"{method} is implemented for Linear layers; the GRU encoder has none"
        cls = {"lora": LoRALinear, "rslora": RSLoRALinear, "lorma": LoRMALinear}[method]
        names = inject(bb, cls, cfg["adapter_targets"], cfg["rank"], cfg["alpha"])
        adapters = [p for n, p in bb.named_parameters() if n.endswith((".A", ".B"))]
        log(f"{method}: rank {cfg['rank']}, alpha {cfg['alpha']}, {len(names)} adapted Linear layers, e.g. {names[:4]}")
    elif method == "bitfit":
        adapters = [p for n, p in bb.named_parameters() if n.endswith("bias") or re.search(r"bias_(ih|hh)", n)]
        assert adapters, "this backbone has no bias terms: BitFit is not applicable (Models 4/5 use bias-free layers)"
        log(f"bitfit: {len(adapters)} bias tensors trainable")
    else:
        assert method in ("full", "kd"), f"unknown method {method}"
    for p in adapters: p.requires_grad = True
    ad_ids = {id(p) for p in adapters}
    backbone_params = [p for p in bb.parameters() if p.requires_grad and id(p) not in ad_ids]
    head = list(model.head.parameters()) + (list(model.attn_pool.parameters()) if model.attn_pool is not None else [])
    return [("head", head, cfg["lr_head"]),
            ("adapter", adapters, cfg["lr_adapter"]),
            ("backbone", backbone_params, cfg["lr_backbone"])]

In [ ]:
# ---- prediction ----
@torch.no_grad()
def case_logits(model, chunks, ids, cfg):
    """document id -> (N_LABELS,) logits."""
    model.eval(); out = {}
    order = sorted(ids, key=lambda d: (len(chunks[d]), max(len(c) for c in chunks[d])))   # similar sizes: less padding
    for grp in case_batches(order, chunks, 2 * cfg["batch_tokens"]):
        x, keep, cc = make_batch(grp, chunks)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            lg = model(x.to(DEVICE), None if keep is None else keep.to(DEVICE), cc.to(DEVICE), len(grp)).float().cpu()
        for b, d in enumerate(grp): out[d] = lg[b]
    return out

def probs(logits, ids):
    return torch.sigmoid(torch.stack([logits[d] for d in ids])).numpy()

In [ ]:
# ---- one experiment ----
RESULTS, TEACHER_LOGITS = [], {}

def count_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

def fixed_subset(ids, n):
    """The same fixed random subset for every run (seed 0), so KD teacher logits cover the student's data."""
    ids = sorted(ids); random.Random(0).shuffle(ids)
    return ids[:n] if n else ids

def run_experiment(run):
    cfg = {**COMMON, **run}
    if SMOKE: cfg.update(epochs=1)
    unset = [k for k, v in cfg.items() if v is ...]
    assert not unset, f"run {cfg['name']}: replace the `...` placeholder with a value for {unset}"
    name = cfg["name"]
    out_dir = os.path.join(WORK, name); os.makedirs(out_dir, exist_ok=True)
    section(f"RUN {name}: model {cfg['model']}, method {cfg['method']}, section {cfg['section']}, "
            f"{cfg['n_chunks']} x {cfg['chunk_len']} tokens, pool {cfg['pool']}")
    log("config:", json.dumps({k: v for k, v in cfg.items() if k not in COMMON or COMMON[k] != v}))
    set_seed(cfg["seed"])
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

    backbone, info = load_backbone(cfg["model"], cfg["student_layers"])
    log(f"checkpoint {info['folder']} (kind {info['kind']}, layers {info['n_layers']}/{info['full_layers']}, "
        f"prefix_lm={info['prefix_lm']}, unused checkpoint tensors {info['unused_keys']})")
    if cfg["model"] in ("m4", "m5"):                     # same file format: make sure it is the right checkpoint
        assert bool(info["prefix_lm"]) == (cfg["model"] == "m5"), (
            f"{cfg['model']} loaded a checkpoint with prefix_lm={info['prefix_lm']} - wrong MODEL_DIRS['{cfg['model']}']?")
    if info["kind"] != "gru":
        assert cfg["chunk_len"] <= info["config"]["max_seq_len"], "chunk_len longer than the model's context"
    toks = get_tokens(info["tokenizer"])

    model = CaseClassifier(backbone, cfg["head_dropout"], causal=(cfg["attn"] == "causal" and info["kind"] != "gru"),
                           pool=cfg["pool"], n_out=N_LABELS)
    train_ids = fixed_subset(TRAIN_ALL, cfg["max_train_docs"])
    n_pos = gold_matrix(train_ids).sum(0)
    with torch.no_grad():                                # head bias = log-odds of each statute in training: the
        f = np.clip(n_pos / len(train_ids), 1e-4, 1 - 1e-4)   # untrained head predicts the training frequencies
        model.head.bias.copy_(torch.tensor(np.log(f / (1 - f)), dtype=torch.float32))
    groups = setup_method(model, cfg)
    model.to(DEVICE)
    n_total, n_train = count_params(model), count_params(model, True)
    log(f"parameters: total {n_total / 1e6:.2f}M, trainable {n_train / 1e6:.3f}M ({100 * n_train / n_total:.2f}%)")

    sel_ids = fixed_subset(VAL_IDS, cfg["dev_select_docs"])
    tr_ch = case_chunks(toks, train_ids, cfg["section"], cfg)
    va_ch = case_chunks(toks, VAL_IDS, cfg["section"], cfg)
    Y_tr = {d: torch.from_numpy(r.astype(np.float32)) for d, r in zip(train_ids, gold_matrix(train_ids))}
    Y_va, Y_sel = gold_matrix(VAL_IDS), gold_matrix(sel_ids)
    n_tok = sum(len(x) for d in train_ids for x in tr_ch[d])
    log(f"train: {len(train_ids)} documents, {n_tok / 1e6:.2f}M tokens/epoch; documents per statute min {n_pos.min()}, "
        f"max {n_pos.max()}; epoch selection on {len(sel_ids)} dev documents")

    teacher = None
    if cfg["method"] == "kd":
        t = cfg["teacher"]
        if t not in TEACHER_LOGITS:
            hits = glob.glob(f"/kaggle/input/**/{t}/teacher_logits.pt", recursive=True)
            assert hits, f"teacher run {t} has not run in this session and no {t}/teacher_logits.pt input was found"
            TEACHER_LOGITS[t] = torch.load(hits[0])
        teacher = TEACHER_LOGITS[t]
        assert all(d in teacher for d in train_ids), "teacher logits do not cover the training documents (other max_train_docs?)"
        log(f"kd: teacher {t}, temperature {cfg['kd_temperature']}, alpha {cfg['kd_alpha']}")

    pos_w = None
    if cfg["class_weighting"] in ("pos_weight", "sqrt_pos_weight"):
        pw = (len(train_ids) - n_pos) / np.maximum(1, n_pos)
        if cfg["class_weighting"] == "sqrt_pos_weight": pw = np.sqrt(pw)
        pos_w = torch.tensor(pw, dtype=torch.float32, device=DEVICE)
        log(f"{cfg['class_weighting']}: min {pw.min():.2f}, median {np.median(pw):.2f}, max {pw.max():.2f}")
    else:
        assert cfg["class_weighting"] == "none", cfg["class_weighting"]
    n_batches = len(case_batches(train_ids, tr_ch, cfg["batch_tokens"]))
    steps = math.ceil(n_batches / cfg["grad_accum"]) * cfg["epochs"]
    pg = [dict(params=[p for p in ps if p.ndim >= 2], lr=lr, weight_decay=cfg["weight_decay"]) for _, ps, lr in groups] + \
         [dict(params=[p for p in ps if p.ndim < 2], lr=lr, weight_decay=0.0) for _, ps, lr in groups]
    opt = torch.optim.AdamW([g for g in pg if g["params"]])
    warm = max(1, int(cfg["warmup_frac"] * steps))
    lr_at = lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / max(1, steps - warm))   # warmup, linear decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))
    log(f"~{n_batches} batches/epoch of <= {cfg['batch_tokens']} tokens, optimizer steps ~{steps}, warmup {warm}")

    best, best_state, history, bad = -1.0, None, [], 0
    rng = random.Random(cfg["seed"])                     # own generator: parallel runs do not disturb each other
    order = list(train_ids)
    t_train = time.time()
    for ep in range(cfg["epochs"]):
        model.train(); t0 = time.time()
        rng.shuffle(order)
        bl = case_batches(order, tr_ch, cfg["batch_tokens"])
        tot, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for bi, grp in enumerate(bl):
            x, keep, cc = make_batch(grp, tr_ch)
            y = torch.stack([Y_tr[d] for d in grp]).to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
                lg = model(x.to(DEVICE), None if keep is None else keep.to(DEVICE), cc.to(DEVICE), len(grp))
            loss = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pos_w)
            if teacher is not None:                      # soft targets: the teacher's sigmoid probabilities at temperature T
                T_ = cfg["kd_temperature"]
                tl = torch.stack([teacher[d].float() for d in grp]).to(DEVICE)
                kd = F.binary_cross_entropy_with_logits(lg / T_, torch.sigmoid(tl / T_)) * T_ * T_
                loss = (1 - cfg["kd_alpha"]) * loss + cfg["kd_alpha"] * kd
            scaler.scale(loss / cfg["grad_accum"]).backward()
            nb += 1; tot += loss.item()
            if nb % cfg["grad_accum"] == 0 or bi == len(bl) - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if bi % 200 == 0 and bi:
                log(f"    batch {bi}/{len(bl)}: loss {tot / nb:.4f}, {n_tok * bi / len(bl) / (time.time() - t0):.0f} tokens/s")
        t_ep = time.time() - t0
        _, vm = best_threshold(probs(case_logits(model, va_ch, sel_ids, cfg), sel_ids), Y_sel)
        history.append(dict(epoch=ep + 1, train_loss=tot / max(1, nb), **{f"val_{k}": v for k, v in vm.items()},
                            train_seconds=t_ep, seconds=time.time() - t0))
        log(f"  epoch {ep + 1:2d}/{cfg['epochs']}: train loss {tot / max(1, nb):.4f} | dev ({len(sel_ids)}) {fmt(vm)} | "
            f"train {t_ep:.0f}s ({n_tok / t_ep:.0f} tokens/s), dev {time.time() - t0 - t_ep:.0f}s")
        if vm["macro_f1"] > best:
            best, bad = vm["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        else:
            bad += 1
            if cfg["patience"] and bad >= cfg["patience"]:
                log("  early stop"); break
    train_secs = time.time() - t_train
    model.load_state_dict(best_state, strict=False)
    best_ep = max(history, key=lambda h: h["val_macro_f1"])["epoch"]

    # full dev at the best epoch (its global threshold is used for test), then other parts of the document
    vl = case_logits(model, va_ch, VAL_IDS, cfg)
    thr, vm = best_threshold(probs(vl, VAL_IDS), Y_va)
    log(f"best epoch {best_ep}: dev ({len(VAL_IDS)}) {fmt(vm)}")
    dev_sections = {f"{cfg['section']}:{cfg['n_chunks']}": vm}
    dev_sec_logits = {}
    for spec in cfg["eval_sections"]:
        t0 = time.time()
        dev_sec_logits[spec] = case_logits(model, case_chunks(toks, VAL_IDS, spec, cfg), VAL_IDS, cfg)
        dev_sections[spec] = best_threshold(probs(dev_sec_logits[spec], VAL_IDS), Y_va)[1]
        log(f"  dev reading {spec:10s}: {fmt(dev_sections[spec])} ({time.time() - t0:.0f}s)")

    # test predictions
    t0 = time.time()
    tl_ = case_logits(model, case_chunks(toks, TEST_IDS, cfg["section"], cfg), TEST_IDS, cfg)
    infer_secs = time.time() - t0
    Z = decide(probs(tl_, TEST_IDS), thr)
    preds = [{"id": d, "pred": [int(l) for l in np.flatnonzero(z)]} for d, z in zip(TEST_IDS, Z)]
    assert len(preds) == len(TEST_IDS) and all(p["pred"] and max(p["pred"]) < N_LABELS for p in preds)
    with open(os.path.join(out_dir, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(preds, f)
    with zipfile.ZipFile(os.path.join(out_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out_dir, "prediction.json"), "prediction.json")
    log(f"test: {len(TEST_IDS)} documents, {Z.sum(1).mean():.2f} statutes per document (thr {thr:.2f}), "
        f"inference {infer_secs:.0f}s")
    test_sec_logits = {}
    for spec in cfg["test_sections"]:
        t0 = time.time()
        test_sec_logits[spec] = case_logits(model, case_chunks(toks, TEST_IDS, spec, cfg), TEST_IDS, cfg)
        log(f"  test logits reading {spec} ({time.time() - t0:.0f}s)")
    half = lambda L: {d: v.half() for d, v in L.items()}
    torch.save(dict(dev=half(vl), test=half(tl_), dev_sections={k: half(v) for k, v in dev_sec_logits.items()},
                    test_sections={k: half(v) for k, v in test_sec_logits.items()}),
               os.path.join(out_dir, "logits.pt"))      # for later post-processing / ensembles

    # teacher logits (train subset + dev), for a KD run in this session or a later one
    if cfg["save_teacher_logits"] or any(r.get("teacher") == name for r in RUNS):
        t0 = time.time()
        TEACHER_LOGITS[name] = half(case_logits(model, tr_ch, train_ids, cfg))
        TEACHER_LOGITS[name].update(half(vl))
        torch.save(TEACHER_LOGITS[name], os.path.join(out_dir, "teacher_logits.pt"))
        log(f"saved teacher logits for {len(TEACHER_LOGITS[name])} documents ({time.time() - t0:.0f}s)")

    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
    res = dict(name=name, model=cfg["model"], method=cfg["method"], section=cfg["section"], n_chunks=cfg["n_chunks"],
               chunk_len=cfg["chunk_len"], pool=cfg["pool"], attn=cfg["attn"] if info["kind"] != "gru" else "bigru",
               layers=f"{info['n_layers']}/{info['full_layers']}", total_params=n_total, trainable_params=n_train,
               train_docs=len(train_ids), best_epoch=best_ep, **{f"val_{k}": v for k, v in vm.items()},
               dev_sections=dev_sections, train_minutes=train_secs / 60, test_inference_seconds=infer_secs,
               peak_gpu_gb=peak, history=history, config=cfg, checkpoint=info["folder"])
    json.dump(res, open(os.path.join(out_dir, "val_metrics.json"), "w"), indent=1)
    Zv = decide(probs(vl, VAL_IDS), thr)
    json.dump([{"id": d, "pred": [int(l) for l in np.flatnonzero(z)]} for d, z in zip(VAL_IDS, Zv)],
              open(os.path.join(out_dir, "val_predictions.json"), "w"))
    if cfg["save_weights"]:
        torch.save(dict(trainable=best_state, config=cfg, backbone_config=info["config"]), os.path.join(out_dir, "trainable_weights.pt"))
    log(f"train {train_secs / 60:.1f} min, peak GPU memory {peak:.2f} GB, outputs in {out_dir}")
    RESULTS.append(res)
    del model, opt, backbone; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

In [ ]:
# ---- run everything: one worker per GPU, each takes the next run from the queue (2 x T4 -> 2 runs at a time) ----
# torch.cuda.set_device is per thread, and DEVICE = "cuda" means "this thread's GPU", so every run stays on its GPU.
# A KD run whose teacher is trained in this same session waits until the teacher has finished.
SELECTED = [r for r in RUNS if not RUN_ONLY or r["name"] in RUN_ONLY]
GPU_IDS = json.loads(os.environ.get("P5_GPU_IDS", "null")) or (list(range(torch.cuda.device_count())) if DEVICE.type == "cuda" else [None])
GPU_IDS = GPU_IDS[:max(1, len(SELECTED))]
_queue, _qlock, _finished = list(SELECTED), threading.Lock(), set()
_in_session = {r["name"] for r in SELECTED}

def _worker(gpu):
    if gpu is not None: torch.cuda.set_device(gpu)
    while True:
        with _qlock:
            if not _queue: return
            ready = [r for r in _queue if r.get("teacher") not in _in_session or r["teacher"] in _finished]
            run = ready[0] if ready else None
            if run: _queue.remove(run)
        if run is None:
            time.sleep(20); continue
        log(f"starting {run['name']}")
        try:
            run_experiment(run)
        except Exception as e:                           # one failing run must not lose the others
            import traceback
            log(f"RUN {run['name']} FAILED: {e!r}\n{traceback.format_exc()}")
            gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        finally:
            with _qlock: _finished.add(run["name"])

section(f"RUNNING {[r['name'] for r in SELECTED]} on {len(GPU_IDS)} worker(s): GPUs {GPU_IDS}")
_threads = [threading.Thread(target=_worker, args=(g,), name=f"gpu{g if g is not None else 'cpu'}-{i}") for i, g in enumerate(GPU_IDS)]
for t in _threads: t.start()
for t in _threads: t.join()

In [ ]:
# ---- summary + one zip to download ----
section("SUMMARY")
log(f"{'run':24s} {'model':5s} {'method':7s} {'input':8s} {'layers':6s} {'total':>8s} {'trainable':>10s} {'best ep':>7s} "
    f"{'macroF1':>7s} {'microF1':>7s} {'subset':>6s} {'thr':>4s} {'train min':>9s} {'infer s':>7s} {'peak GB':>7s}")
for r in RESULTS:
    inp = f"{r['section']}:{r['n_chunks']}"
    log(f"{r['name']:24s} {r['model']:5s} {r['method']:7s} {inp:8s} {r['layers']:6s} {r['total_params'] / 1e6:7.1f}M "
        f"{r['trainable_params'] / 1e6:9.3f}M {r['best_epoch']:7d} {r['val_macro_f1']:7.4f} {r['val_micro_f1']:7.4f} "
        f"{r['val_subset_acc']:6.4f} {r['val_threshold']:4.2f} {r['train_minutes']:9.1f} {r['test_inference_seconds']:7.0f} "
        f"{r['peak_gpu_gb']:7.2f}")
log("dev macro-F1 reading other parts of the document (each with its own best global threshold):")
for r in RESULTS:
    log(f"  {r['name']:24s} " + " ".join(f"{k}={v['macro_f1']:.4f}" for k, v in r["dev_sections"].items()))

json.dump(RESULTS, open(os.path.join(WORK, "phase5_results_summary.json"), "w"), indent=1)
_log.flush()
bundle = os.path.join(WORK, "phase5_results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(WORK, "**", "*"), recursive=True):
        if (f.endswith((".txt", ".json", ".zip", "logits.pt"))          # teacher_logits.pt: reusable as a KD input
                and not f.endswith("phase5_results_bundle.zip")):
            z.write(f, os.path.relpath(f, WORK))
log(f"bundle: {bundle} ({os.path.getsize(bundle) / 1e6:.1f} MB)")